In [1]:
import numpy as np
from scipy.spatial.transform import Rotation
from h5_utils import load

In [3]:
# 1. Define the file path
FILE_PATH = '6_TRAJECTORY_AND_SONAR.h5'

# 2. Load the Trajectory (H)
TRAJ = load(FILE_PATH, 'TRAJ')
H    = TRAJ.H       # Shape: (N, 7) - [x, y, z, qw, qx, qy, qz]
TIME = TRAJ.TIME    # Timestamps for the poses

# 3. Load Vertical Sonar Data (M^v)
M_v_data = load(FILE_PATH, 'M_v')
rho_v      = M_v_data.rho       # Distance/Range measurements
psi_v      = M_v_data.psi       # Rotation angles
pose_idx_v = M_v_data.pose_idx.astype(int)  # The mapping function \sigma^v(i)

# 4. Load Horizontal Sonar Data (M^h)
M_h_data = load(FILE_PATH, 'M_h')
# M_h_data.M is a (3, total_returns) array where:
# Row 0 = rho_h, Row 1 = theta_h (NaNs to be estimated), Row 2 = psi_h
rho_h      = M_h_data.M[0, :]   
theta_h    = M_h_data.M[1, :]   # Currently NaNs
psi_h      = M_h_data.M[2, :]   
pose_idx_h = M_h_data.pose_idx.astype(int)  # The mapping function \sigma^h(i)

Loaded TRAJ from 6_TRAJECTORY_AND_SONAR.h5
Loaded M_v from 6_TRAJECTORY_AND_SONAR.h5
Loaded M_h from 6_TRAJECTORY_AND_SONAR.h5


In [ ]:
# ---------------------------------------------------------------------------
# Helpers: SE(3) ↔ matrix conversions
# ---------------------------------------------------------------------------

def pose_to_SE3(pose):
    """
    Convert a 7-vector pose [x, y, z, qw, qx, qy, qz] to a 4×4 SE(3) matrix.

    The notebook stores poses as:
        H = np.hstack((X_IMU_ARRAY, QUAT_IMU_ARRAY))   # shape (N, 7)
        where QUAT_IMU is [qw, qx, qy, qz]  (scalar-first convention).
    """
    t              = pose[:3]          # translation [x, y, z]
    qw, qx, qy, qz = pose[3:]          # quaternion (scalar first)
    R              = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()   # 3×3 # scipy.spatial.transform.Rotation uses scalar-last [qx, qy, qz, qw]

    T         = np.eye(4)
    T[:3, :3] = R
    T[:3,  3] = t
    return T


def SE3_log(T):
    """
    SE(3) logarithm map.

    Given a 4×4 transformation matrix T = [R | t ; 0 | 1],
    returns the 6-vector twist  ξ = [v ; ω]  in  se(3).

    The rotation part ω is the axis-angle vector (‖ω‖ = rotation angle).
    The translation part v satisfies  t = V ω  where V is the left-Jacobian
    of SO(3).

    Reference: Eq. (13) in the paper, plus standard SE(3) Lie theory.
    """
    R = T[:3, :3]
    t = T[:3,  3]

    # --- rotation part: ω (axis-angle) ---
    omega = Rotation.from_matrix(R).as_rotvec()   # 3-vector; ‖ω‖ = angle θ
    theta = np.linalg.norm(omega)

    # --- translation part: v = V⁻¹ t ---
    if theta < 1e-9:
        # First-order approximation when rotation is negligible
        v = t.copy()
    else:
        # Skew-symmetric matrix of ω (not unit vector)
        W = np.array([
            [ 0,        -omega[2],  omega[1]],
            [ omega[2],  0,        -omega[0]],
            [-omega[1],  omega[0],  0       ]
        ])
        # Left Jacobian of SO(3):
        #   V = I  +  (1 - cosθ)/θ²  ·  W  +  (θ - sinθ)/θ³  ·  W²
        V = (np.eye(3)
             + (1 - np.cos(theta)) / theta**2 * W
             + (theta - np.sin(theta)) / theta**3 * (W @ W))

        v = np.linalg.solve(V, t)   # v = V⁻¹ t

    return np.concatenate([v, omega])   # 6-vector [v; ω]


# ---------------------------------------------------------------------------
# Algorithm 1 — Line 1:  s = s(H)
# ---------------------------------------------------------------------------

def compute_curvilinear_abscissa(H):
    """
    Compute the curvilinear abscissa for every pose in the trajectory.

    Implements Equation 12  s(t) = L_{η(0)}^{η(t)}
    via the discrete approximation of Equation 13:

        s_i = Σ_{k=1}^{i}  sqrt( log(T_{k-1}⁻¹ T_k)ᵀ G log(T_{k-1}⁻¹ T_k) )

    where the 6×6 matrix G = diag(I₃, 2·I₃)  (Eq. 10) weights translations
    by 1 and rotations by 2, yielding the norm:

        ‖ξ‖_G  =  sqrt( ‖v‖² + 2‖ω‖² )

    Parameters
    ----------
    H : np.ndarray, shape (N, 7)
        Trajectory array.  Each row is [x, y, z, qw, qx, qy, qz].

    Returns
    -------
    s : np.ndarray, shape (N,)
        Cumulative arc length (curvilinear abscissa) for each pose.
        s[0] = 0 by definition (Eq. 12, length from the start).
    """
    N = H.shape[0]
    s = np.zeros(N)   # s[0] = 0 by definition

    # G is block-diagonal: translation weight = 1, rotation weight = 2
    # ‖ξ‖_G² = ξᵀ G ξ = ‖v‖² + 2‖ω‖²
    # We compute this directly rather than building the 6×6 matrix.

    for i in range(1, N):
        T_prev   = pose_to_SE3(H[i - 1])   # T_{k-1}
        T_curr   = pose_to_SE3(H[i])       # T_k

        # Relative (incremental) transformation: ΔT = T_{k-1}⁻¹ · T_k
        # For SE(3): inverse is  [Rᵀ | -Rᵀt ; 0 | 1]
        T_rel    = np.linalg.inv(T_prev) @ T_curr
        xi       = SE3_log(T_rel)  # SE(3) log map → 6-vector [v ; ω]
        v, omega = xi[:3], xi[3:]
        delta_s  = np.sqrt(np.dot(v, v) + 2.0 * np.dot(omega, omega)) # Weighted norm  sqrt(‖v‖² + 2‖ω‖²)  — direct expansion of ξᵀ G ξ
        s[i]     = s[i - 1] + delta_s # Accumulate  (Eq. 13)

    return s


# ---------------------------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Minimal synthetic example: straight-line translation of 1 m/step, no rotation
    N = 5
    H_test = np.zeros((N, 7))
    H_test[:, 0] = np.arange(N) * 1.0   # x increases by 1 m each step
    H_test[:, 3] = 1.0                  # qw = 1  (identity rotation)

    s = compute_curvilinear_abscissa(H_test)
    print("Poses (x only):", H_test[:, 0])
    print("Curvilinear abscissa s:", s)
    # Expected: [0, 1, 2, 3, 4]  (purely translational, ‖v‖=1, ‖ω‖=0)
    assert np.allclose(s, np.arange(N), atol=1e-9), "Sanity check failed!"
    print("Sanity check passed ✓")

Poses (x only): [0. 1. 2. 3. 4.]
Curvilinear abscissa s: [0. 1. 2. 3. 4.]
Sanity check passed ✓


In [5]:
# --- Algorithm 1, Line 1: Compute curvilinear abscissa for the whole trajectory ---
s_traj = compute_curvilinear_abscissa(H)